# Patient-level cohort construction and split

Builds the six diagnostic-group folders used in the paper from the original label files:
1. keep patients aged 65+ (CheXpert: frontal views only);
2. map source labels to study groups by priority: Pneumonia > Cardiomegaly > Pleural/Extrapulmonary/Volume > Airspace Opacity (non-infectious) > Focal Lesion/Chronic Change > No Finding (images with none of these are dropped; see `data/label_mapping.csv`);
3. split **patients** 60/20/20 into Train/Validation/Test separately within each dataset (`random_state=42`);
4. copy the images into `OUTPUT_ROOT/<Split>/<Group>/`.

The logic is unchanged from the run used for the paper; only the file paths were turned into variables. The folder `Airspace Opacity (non-infectious)` was later renamed `Weakly_Supervised_Uncertain`, and files were renamed to the `<source>_<patient>_<age>_<sex>_<view>_...` pattern read by the training notebook (see the renaming script).

In [52]:
import pandas as pd
import os
import shutil
from sklearn.model_selection import train_test_split

In [53]:
# ---------------------------------------------------------------------------
# Paths: edit these for your machine (originally run on Windows with local drives).
#   CHEXPERT_CSV        : train_visualCheXbert.csv (VisualCheXbert image-level labels, CheXpert v1.0)
#   CHESTX8_CSV         : Data_Entry_2017_v2020.csv (NIH ChestX-ray8/14 labels)
#   CHEXPERT_IMAGE_ROOT : folder containing CheXpert-v1.0 (train/patientXXXXX/studyN/viewN_frontal.jpg)
#   CHESTX8_IMAGE_ROOT  : folder containing the extracted NIH images
#   OUTPUT_ROOT         : where the Train/Validation/Test group folders are created
# ---------------------------------------------------------------------------
DATA_DIR = os.environ.get("GPPM_DATA_DIR", "./data_raw")

CHEXPERT_CSV = os.path.join(DATA_DIR, "CheXpert", "train_visualCheXbert.csv")
CHESTX8_CSV = os.path.join(DATA_DIR, "ChestX-ray8", "Data_Entry_2017_v2020.csv")
CHEXPERT_IMAGE_ROOT = os.path.join(DATA_DIR, "CheXpert", "CheXpert-v1.0")
CHESTX8_IMAGE_ROOT = os.path.join(DATA_DIR, "ChestX-ray8", "images")

OUTPUT_ROOT = os.environ.get("GPPM_OUTPUT_DIR", "./Patient_Level_Split_Enforced")

In [54]:
class DatasetOrganizer:
    def __init__(self):
        self.stats = {'CheXpert': 0, 'ChestX-ray8': 0}

    def get_split_map(self, unique_ids):
        """Creates a patient-level 60-20-20 mapping."""
        train_pts, temp_pts = train_test_split(unique_ids, test_size=0.40, random_state=42)
        val_pts, test_pts = train_test_split(temp_pts, test_size=0.50, random_state=42)
        
        mapping = {p: 'Train' for p in train_pts}
        mapping.update({p: 'Validation' for p in val_pts})
        mapping.update({p: 'Test' for p in test_pts})
        return mapping

    def get_chex_label(self, row):
        """Priority logic: Pneumonia > Cardiomegaly > Others."""
        if row['Pneumonia'] == 1: return "Pneumonia (Infectious)"
        if row['Cardiomegaly'] == 1: return "Cardiomegaly"
        
        if any(row[col] == 1 for col in ['Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other']):
            return "Pleural/Extrapulmonary/Volume Disease"
        if row['Edema'] == 1: 
            return "Airspace Opacity (non-infectious)"
        if row['No Finding'] == 1: 
            return "No Finding"
        return None

    def get_cxr8_label(self, label_str):
        """Priority logic for CXR8 piped strings."""
        labels = [l.strip().lower() for l in str(label_str).split('|')]
        if 'pneumonia' in labels: return "Pneumonia (Infectious)"
        if 'cardiomegaly' in labels: return "Cardiomegaly"
        
        if any(l in labels for l in ['atelectasis', 'effusion', 'hernia', 'pleural_thickening', 'pneumothorax']):
            return "Pleural/Extrapulmonary/Volume Disease"
        if any(l in labels for l in ['consolidation', 'edema', 'infiltration']):
            return "Airspace Opacity (non-infectious)"
        if any(l in labels for l in ['emphysema', 'fibrosis', 'mass', 'nodule']):
            return "Focal Lesion/Chronic Change"
        if 'no finding' in labels: 
            return "No Finding"
        return None

    def print_full_report(self, df_chex, df_cxr8):
        """Generates the full demographic, age, and class distribution report."""
        print("\n" + "="*60)
        print("                  DATASET STATISTICAL REPORT")
        print("="*60)
        
        # 1. Prepare Combined Patient Data (Unique IDs only)
        combined_pts = pd.concat([
            df_chex.drop_duplicates('PatientID')[['PatientID', 'Sex', 'Age', 'Split']].rename(columns={'Sex': 'Gender'}),
            df_cxr8.drop_duplicates('PatientID')[['PatientID', 'Patient Sex', 'Patient Age', 'Split']].rename(columns={'Patient Sex': 'Gender', 'Patient Age': 'Age'})
        ])

        # 2. Add Age Bins
        bins = [65, 76, 86, 96, 111]
        labels = ['65-75', '76-85', '86-95', '96-106']
        combined_pts['AgeGroup'] = pd.cut(combined_pts['Age'], bins=bins, labels=labels, right=False)

        print("\n1. UNIQUE PATIENT GENDER DISTRIBUTION (COUNTS)")
        print(combined_pts.groupby(['Split', 'Gender']).size().unstack(fill_value=0))

        print("\n2. UNIQUE PATIENT COUNTS PER AGE GROUP")
        print(combined_pts.groupby(['Split', 'AgeGroup']).size().unstack(fill_value=0))

        print("\n3. IMAGE TYPE (AP/PA) DISTRIBUTION PER SPLIT")
        chex_views = df_chex.groupby(['Split', 'AP/PA']).size().unstack(fill_value=0)
        cxr8_views = df_cxr8.groupby(['Split', 'View Position']).size().unstack(fill_value=0)
        print(f"CheXpert Views:\n{chex_views}\n\nChestX-ray8 Views:\n{cxr8_views}")

        # 4. NEW: CLASS DISTRIBUTION REPORT
        print("\n4. IMAGE DISTRIBUTION BY DIAGNOSIS GROUP")
        combined_images = pd.concat([
            df_chex[['Target_Folder', 'Split']], 
            df_cxr8[['Target_Folder', 'Split']]
        ])
        class_dist = combined_images.groupby(['Target_Folder', 'Split']).size().unstack(fill_value=0)
        print(class_dist[['Train', 'Validation', 'Test']])
        
        # Calculate Imbalance for the User
        print("\n(Note: If one class is significantly lower than others, consider Weighted Loss or Augmentation)")

    def _move(self, df, source_root, dataset_type):
        """Physically moves and renames files."""
        for _, row in df.iterrows():
            dest_dir = os.path.join(OUTPUT_ROOT, row['Split'], row['Target_Folder'])
            os.makedirs(dest_dir, exist_ok=True)
            
            if dataset_type == "CheXpert":
                parts = row['Path'].split('/')
                new_name = f"CheXpert_{parts[2]}_{parts[3]}_{parts[4]}"
                src_path = os.path.join(source_root, *parts[1:])
            else:
                new_name = f"CXR8_{row['Image Index']}"
                src_path = os.path.join(source_root, row['Image Index'])

            try:
                if os.path.exists(src_path):
                    shutil.copy2(src_path, os.path.join(dest_dir, new_name))
                    self.stats[dataset_type] += 1
            except Exception:
                pass

    def run(self):
        # 1. Load and Filter
        df_chex = pd.read_csv(CHEXPERT_CSV)
        df_cxr8 = pd.read_csv(CHESTX8_CSV)

        df_chex = df_chex[(df_chex['Age'] >= 65) & (df_chex['Frontal/Lateral'].str.lower() == 'frontal')].copy()
        df_cxr8 = df_cxr8[df_cxr8['Patient Age'] >= 65].copy()

        # 2. Patient-Level Split Mapping
        df_chex['PatientID'] = df_chex['Path'].str.split('/').str[2]
        df_cxr8['PatientID'] = df_cxr8['Patient ID']
        
        chex_map = self.get_split_map(df_chex['PatientID'].unique())
        cxr8_map = self.get_split_map(df_cxr8['Patient ID'].unique())
        
        df_chex['Split'] = df_chex['PatientID'].map(chex_map)
        df_cxr8['Split'] = df_cxr8['Patient ID'].map(cxr8_map)

        # 3. Labeling based on Priority
        df_chex['Target_Folder'] = df_chex.apply(self.get_chex_label, axis=1)
        df_cxr8['Target_Folder'] = df_cxr8['Finding Labels'].apply(self.get_cxr8_label)

        df_chex_final = df_chex.dropna(subset=['Target_Folder'])
        df_cxr8_final = df_cxr8.dropna(subset=['Target_Folder'])

        # 4. Reporting
        self.print_full_report(df_chex_final, df_cxr8_final)

        # 5. File Movement
        self._move(df_chex_final, CHEXPERT_IMAGE_ROOT, "CheXpert")
        self._move(df_cxr8_final, CHESTX8_IMAGE_ROOT, "ChestX-ray8")

In [ ]:
if __name__ == "__main__":
    organizer = DatasetOrganizer()
    organizer.run()


                  DATASET STATISTICAL REPORT

1. UNIQUE PATIENT GENDER DISTRIBUTION (COUNTS)
Gender        F  Female     M  Male
Split                              
Test        328    2554   503  2845
Train       939    7690  1552  8587
Validation  339    2578   492  2826

2. UNIQUE PATIENT COUNTS PER AGE GROUP
AgeGroup    65-75  76-85  86-95  96-106
Split                                  
Test         3174   2011    951      94
Train        9468   6135   2897     268
Validation   3231   1975    938      91

3. IMAGE TYPE (AP/PA) DISTRIBUTION PER SPLIT
CheXpert Views:
AP/PA          AP  LL    PA
Split                      
Test        14208   2  1900
Train       41139   3  5515
Validation  13930   1  1910

ChestX-ray8 Views:
View Position    AP    PA
Split                    
Test           1183  1760
Train          3530  5757
Validation     1391  1926

4. IMAGE DISTRIBUTION BY DIAGNOSIS GROUP
Split                                  Train  Validation  Test
Target_Folder                